In [1]:
import numpy as np
import pandas as pd
from transformers  import AutoTokenizer ,DistilBertTokenizerFast ,pipeline
import string 
import torch
import torch.nn as nn

/home/anirban/all_pro/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# classifier = pipeline(
#     task="text-classification",
#     model="bert-case-uncased",
#     # dtype=torch.float16,
#     device=0
# )

# result = classifier("I love using Hugging Face Transformers!")
# print(result)

In [3]:
df=pd.read_csv('data_for_preprocessing.csv')

In [4]:
df['text']=df['Text']
df['author']=df['Author']
df=df.drop(['Unnamed: 0','Text',"Author"],axis=1)

In [5]:
translator = str.maketrans('', '', string.punctuation)
df['text']=df['text'].apply(lambda x: x.translate(translator))
df['text']=df['text'].str.lower()

In [6]:
Tokenizer=AutoTokenizer.from_pretrained('bert-base-uncased')

In [7]:
Tokenizer.vocab_size

30522

In [8]:
texts = df['text'].fillna('').astype(str).tolist()

encodings = Tokenizer(
    texts,
    truncation=True,
    padding='max_length',
    max_length=128,
    # return_attention_mask=True
    return_tensors="pt"
)

input_ids = encodings["input_ids"]

In [9]:
df['author']=df['author'].map({'AI':0,'Human':1})
# df['text']=input_ids

In [10]:
df['text'].head()

0    this study investigates the chemical compositi...
1    this study explores the cultural history of oi...
2     isolation of human peripheral blood mononucle...
3     dynamic bayesian networks dbns are probabilis...
4     within volleyball performance analysis is emp...
Name: text, dtype: str

In [11]:
text=df.copy()

In [12]:
# Store each tokenized sequence as a Python list in one DataFrame row
text['text'] = input_ids.tolist()

In [13]:

class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, input_ids):
        x = self.embedding(input_ids)
        _, h_n = self.rnn(x)
        return self.classifier(h_n[-1])

In [14]:
model = RNNClassifier(vocab_size=Tokenizer.vocab_size, embed_dim=128, hidden_dim=64, num_classes=1)

In [15]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [16]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

# Train/test split (80% train, 20% test)
train_ids, test_ids, train_labels, test_labels = train_test_split(
    input_ids, 
    df['author'].values, 
    test_size=0.2, 
    random_state=42
)

print(f"Train set: {train_ids.shape[0]} samples")
print(f"Test set: {test_ids.shape[0]} samples")

Train set: 4855 samples
Test set: 1214 samples


In [17]:
class TextDataset(Dataset):
    def __init__(self, input_ids, labels):
        self.input_ids = input_ids
        self.labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)  # Shape: [N, 1]
    
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.labels[idx]

# Create datasets
train_dataset = TextDataset(train_ids, train_labels)
test_dataset = TextDataset(test_ids, test_labels)

# Create DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

Train batches: 152
Test batches: 38


In [18]:
def train_epoch(model, train_loader, criterion, optimizer):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for input_ids, labels in train_loader:
        optimizer.zero_grad()
        
        # Forward pass
        logits = model(input_ids)
        loss = criterion(logits, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Compute accuracy
        preds = torch.sigmoid(logits) > 0.5
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    avg_loss = total_loss / len(train_loader)
    accuracy = correct / total
    return avg_loss, accuracy

def evaluate(model, test_loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for input_ids, labels in test_loader:
            logits = model(input_ids)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            preds = torch.sigmoid(logits) > 0.5
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    avg_loss = total_loss / len(test_loader)
    accuracy = correct / total
    return avg_loss, accuracy

# Train the model
num_epochs = 5
for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    test_loss, test_acc = evaluate(model, test_loader, criterion)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train - Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}")
    print(f"  Test  - Loss: {test_loss:.4f}, Accuracy: {test_acc:.4f}")

Epoch 1/5
  Train - Loss: 0.0644, Accuracy: 0.9874
  Test  - Loss: 0.0289, Accuracy: 0.9934
Epoch 2/5
  Train - Loss: 0.0328, Accuracy: 0.9928
  Test  - Loss: 0.0371, Accuracy: 0.9926
Epoch 3/5
  Train - Loss: 0.0261, Accuracy: 0.9946
  Test  - Loss: 0.0340, Accuracy: 0.9926
Epoch 4/5
  Train - Loss: 0.0245, Accuracy: 0.9946
  Test  - Loss: 0.0359, Accuracy: 0.9934
Epoch 5/5
  Train - Loss: 0.0187, Accuracy: 0.9961
  Test  - Loss: 0.0213, Accuracy: 0.9934


In [19]:
def predict_text(text_input):
    
    model.eval()
    
    # Preprocess text (same as training)
    processed_text = text_input.translate(translator).lower()
    
    # Tokenize
    encoding = Tokenizer(
        processed_text,
        truncation=True,
        padding='max_length',
        max_length=128,
        return_tensors="pt"
    )
    
    input_ids_pred = encoding['input_ids']
    
    # Get prediction
    with torch.no_grad():
        logit = model(input_ids_pred)
        probability = torch.sigmoid(logit).item()
        prediction = 1 if probability > 0.5 else 0
        label = "Human" if prediction == 1 else "AI"
    
    return prediction, probability, label


# Test with sample texts
test_texts = [
    "The weather today is beautiful and sunny, perfect for outdoor activities.",
    "Machine learning models require large amounts of data for training and validation purposes."
]

for i, text in enumerate(test_texts, 1):
    pred, conf, label = predict_text(text)
    print(f"Sample {i}:")
    print(f"Text: {text}")
    print(f"Prediction: {label}")
    print(f"Confidence: {conf:.4f}\n")

Sample 1:
Text: The weather today is beautiful and sunny, perfect for outdoor activities.
Prediction: AI
Confidence: 0.0028

Sample 2:
Text: Machine learning models require large amounts of data for training and validation purposes.
Prediction: AI
Confidence: 0.0026



In [20]:
# Interactive prediction - Enter your own text here
interactive_text = input("Enter text to classify (or press Enter to skip): ")

if interactive_text.strip():
    pred, conf, label = predict_text(interactive_text)
    print("\n" + "="*50)
    print(f"Input: {interactive_text}")
    
    print(f"Prediction: {label}")
    print(f"Confidence: {conf:.2%}")
    print("="*50)
else:
    print("No text provided. Skipping prediction.")

No text provided. Skipping prediction.


In [22]:
import pickle
with open('scaler.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('translator.pkl', 'wb') as f:
    pickle.dump(translator, f)
    
with open('encodings.pkl', 'wb') as f:
    pickle.dump(encodings, f)
